# Population Clustering with PCA — Guided Exercises

**AMLG | Genetic variation, SNP encoding, dimensionality reduction, and population visualization**

In this guided notebook, you will explore how genetic differences between individuals can be represented numerically and visualized using **principal component analysis (PCA)**. We start with SNP data, build a simple binary feature matrix, and examine how the first three principal components relate to population labels and sex labels.

### Learning objectives
By the end, you should be able to:
1. Explain what a SNP, consensus base, and binary aberration feature mean **in this notebook**.
2. Convert a table of genetic bases into a numerical feature matrix.
3. Standardize features, run PCA, and interpret a 2D scatter plot.
4. Inspect a PCA loading vector to identify features contributing to a component.
5. Distinguish **visual patterns** from strong biological or causal conclusions.

### How to use this notebook
- Run cells from top to bottom using **Shift + Enter**. Keep the original repository folder structure.
- At each **❓ Exercise**, first read **Task**, **Hints**, **What this teaches**, and **Expected outcome**.
- Write your solution where you see `# TODO`. Starter code is intentionally incomplete.
- A **commented reference answer** is included after `# --- Solution (uncomment to check) ---` in each exercise cell. Attempt it yourself before revealing it.
- Where starter code contains `pass`, **remove `pass`** when you uncomment the relevant answer.
- Some exercises produce plots, not a single numeric answer. Compare the *pattern* with the description rather than expecting pixel-identical plots.

**Learning route:** Load population SNP data → consensus per locus → binary SNP differences → standardize → PCA → population and sex plots → inspect PCA feature loadings.

> **Scope:** We use an intentionally simple **binary mismatch encoding** (base differs from the consensus: yes/no), not a full population-genetics genotype model. PCA is an exploratory visualization tool: apparent clusters do not define discrete biological races, and observed associations do not alone establish causation.

# **Population Clustering Using Principal Component Analysis**

## 1. Load the data

**Concept — genotype table:** Each row represents an individual. The first three columns are **metadata** (individual ID, sex label, and population label); the following columns contain SNP observations at matching loci across individuals.

**Before running:** You should be in the supplied repository environment. The setup cells below preserve the **original paths and download URL** unchanged. If the data already exist locally, they are still read from `data/1000_genomes_data.txt`.

First, let's download the data for this notebook.

In [ ]:
import os
import sys

# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

In [ ]:
from utils import download_file

if not os.path.exists(path="data"):
    os.makedirs(name="data")

download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494344",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fpopulation_clustering%2F1000_genomes_data.txt&dl=1",
    save_filename="data/1000_genomes_data.txt",
)

# Data

The contents in this notebook are motivated by the article [Genes mirror geography within Europe](https://doi.org/10.1038/nature07331).

> A copy of the article is available [here](https://dataverse.harvard.edu/api/access/datafile/10494343) or [here](https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fpopulation_clustering%2Fnature07331.pdf).

In this article, the authors characterize genetic variation in a sample of 3,000 European individuals genotyped at over half a million variable DNA sites in the human genome.
They use principal components analysis (PCA) to produce a two-dimensional visual summary of the observed genetic variation.
The resulting figure bears a notable resemblance to a geographic map of Europe.
Individuals from the same geographic region cluster together and major populations are distinguishable.

<img src="images/pcs_mirror_eu_populations.png" width=600 />

In this notebook, we aim at performing a similar analysis with data from the [1000 Genomes Project](https://www.internationalgenome.org/1000-genomes-summary/).
In the file `1000_genomes_data.txt` (in the `data/` folder) we provide genetic data for 995 individuals with African ancestry.
Each of the 995 lines in the file represents an individual.

The first three columns exhibit the following semantics:

- The first column contains the individual's unique identifier.
- The second column contains the individual's sex (1=male, 2=female).
- The third column represents the population the individual belongs to. There are seven populations in the dataset: ACB, ASW, ESN, GWD, LWK, MSL, YRI.

The populations are described as follows.

> All (other) population descriptions are provided on the [_The International Genome Sample Resource_ website](https://www.internationalgenome.org/cell-lines-and-dna-coriell).

| Population code | Population description                  |
|-----------------|-----------------------------------------|
| ACB             | African Caribbean in Barbados           |
| ASW             | African Ancestry in Southwest US        |
| ESN             | Esan in Nigeria                         |
| GWD             | Gambian in Western Division, The Gambia |
| LWK             | Luhya in Webuye, Kenya                  |
| MSL             | Mende in Sierra Leone                   |
| YRI             | Yoruba in Ibadan, Nigeria               |

The remaining 10,101 columns of each line are single-nucleotide polymorphisms (SNPs) from the individual's genome.
Note: the loci of the SNPs are unknown, but they are the same across all individuals!

We can use the function [`read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) from [pandas](https://pandas.pydata.org) to read in the data from the file `1000_genomes_data.txt`.

### Quick data check

The original loading cell reports the number of individuals and labels. After running it, check that the matrix has **995 rows** and **10,104 columns** (three metadata columns + 10,101 SNP columns). A shape is always written as `(number_of_rows, number_of_columns)`.

**Think:** Why must SNP loci be in the same column for every individual? Because otherwise comparing individuals position by position would be meaningless.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(
    filepath_or_buffer="data/1000_genomes_data.txt", header=None, sep=r"\s+"
)
print(f"Dataframe shape: {df.shape}")

identifiers = df[0]
sexes = df[1]
populations = df[2]

print(f"Number of unique individuals: {len(np.unique(identifiers))}")
print(f"Sexes: {np.unique(sexes)}")
print(f"Populations: {np.unique(populations)}")

# Preprocessing

## 2. Preprocessing: from DNA bases to numbers

**Concept — SNP (single-nucleotide polymorphism):** a genomic position where DNA sequence varies among individuals. In this dataset, columns after the first three contain the observed base at one corresponding position for each individual.

**Concept — consensus base:** For one SNP column, the most frequently occurring base among the individuals. It is a **dataset majority**, not necessarily the evolutionary ancestral allele or a biological reference genome base.

**Example:** If a locus contains `A, A, G, A, G`, its consensus is `A`. The binary mismatch values become `0, 0, 1, 0, 1`. Repeat this for each SNP column to create a matrix with individuals as rows and SNPs as columns.

**Note:** We use a binary encoding for teaching. It loses information about *which* alternative base is present and should not be confused with conventional diploid genotype dosage (0/1/2).

To understand what distinguishes individuals from different populations, we will focus on the differences between the SNP "vectors".
Hence, we first need to compute the consensus bases for all SNP loci.
Then, in a second step, we can compute for each individual the SNP aberrations w.r.t. to the consensus bases.

##### ❓ Computing the consensus bases

### ❓ Exercise 1 — Consensus bases

**Task:** Remove the three metadata columns and compute the most common base **for each of the 10,101 SNP loci**. Store the answers in `consensus_bases` (a NumPy array).

**Hints:**
- `df.drop(columns=[0, 1, 2])` drops metadata without modifying `df`.
- `DataFrame.mode(axis=0)` calculates the mode of each column; `.iloc[0]` selects one mode if tied.
- `.to_numpy()` converts a pandas row into a NumPy array.

**Expected outcome:** `consensus_bases.shape == (10101,)`. Each element is a DNA-base value from its SNP column.

**What this teaches:** How to summarize a feature across samples, and why metadata should not be treated as SNP features.

1. Use the [`drop()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html) function to drop the first three columns (identifiers, sexes, populations) from the dataframe `df`.
2. Then, for each SNP locus, compute the respective consensus base, i.e., the base that appears most often across all individuals. The consensus bases shall be stored in a variable named `consensus_bases` of type `numpy.ndarray` with shape `(10101,)`.

In [ ]:
# TODO: Create snp_df by dropping columns 0, 1, and 2.
# TODO: Find the most frequent base per SNP column.
# TODO: Store the result in consensus_bases as a NumPy array.

# --- Solution (uncomment to check) ---
# snp_df = df.drop(columns=[0, 1, 2])
# consensus_bases = snp_df.mode(axis=0).iloc[0].to_numpy()
# print("Consensus shape:", consensus_bases.shape)
# assert consensus_bases.shape == (10101,)

##### ❓ Infer individual SNP aberrations

### ❓ Exercise 2 — Binary SNP differences

**Task:** Make `snp_matrix` with shape `(995, 10101)` and a binary matrix `X`: `0` if a base matches that locus's consensus, otherwise `1`.

**Hints:**
- `snp_df.to_numpy()` turns the SNP table into a matrix.
- NumPy *broadcasting* lets you compare a `(995, 10101)` matrix with a `(10101,)` vector column by column.
- `np.where(condition, 0, 1)` assigns zeros when a match is true and ones otherwise.

**Expected outcome:** `X.shape == (995, 10101)`; `X` contains only `0` and `1`.

**What this teaches:** How to encode categorical DNA bases as numerical features usable by PCA.

1. Extract a $995 \times 10101$ SNP matrix from the dataframe `df` using [to_numpy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html).
2. Then, compute the aberrations matrix $\mathbf{X}$ such that $x_{i,j} = 0$ if the $i$-th individual's SNP at position $j$ is equal to the $j$-th consensus base, and $x_{i,j} = 1$ if the $i$-th individual's SNP at position $j$ is different from the $j$-th consensus base.

> Hint: [numpy.where()](https://numpy.org/doc/stable/reference/generated/numpy.where.html) might be handy.

In [ ]:
# TODO: Extract the SNP columns as a NumPy matrix.
# TODO: Compare every base with the consensus for its column.
# TODO: Save a binary array named X (0 = match, 1 = mismatch).

# --- Solution (uncomment to check) ---
# snp_matrix = df.drop(columns=[0, 1, 2]).to_numpy()
# X = np.where(snp_matrix == consensus_bases, 0, 1)
# print("Binary feature matrix:", X.shape)
# print("Possible values:", np.unique(X))
# assert X.shape == (995, 10101)
# assert set(np.unique(X)).issubset({0, 1})

# PCA

## 3. PCA: from 10,101 features to a few coordinates

**Intuition:** Each individual is now a point in a space with 10,101 dimensions. **PCA** finds new perpendicular directions (principal components) that capture the greatest remaining variation in the standardized data.

- **PC1** captures the largest direction of variation, **PC2** the next largest orthogonal direction, and **PC3** the next.
- PCA does **not** use the provided population or sex labels to fit its axes: it is **unsupervised**.
- **Standardization** rescales each SNP feature by subtracting its mean and dividing by its standard deviation. For features constant across people, scikit-learn handles their zero variance safely.
- PCA scores (`Z`) tell us where **individuals** lie along the PC axes. PCA components/loadings (`pca.components_`) describe the contribution of **features** to those axes.

**Why standardize?** A binary SNP's variance depends on how common its mismatches are. Scaling changes the relative influence of common and rare binary features; this is a classroom modeling choice, not a universally optimal genotype-PCA preprocessing procedure.

In the next step, our goal is to project the binary feature vectors (rows of $\mathbf{X}$) to a lower-dimensional subspace.

##### ❓ Dimensionality reduction

### ❓ Exercise 3 — Standardize and reduce to 3 dimensions

**Task:** Apply `StandardScaler` to the binary matrix `X`, fit `PCA(n_components=3)`, and store the projected samples in `Z`. Keep the fitted PCA model in `pca` for the last exercise.

**Hints:**
- `X_scaled = StandardScaler().fit_transform(X)`
- `pca = PCA(n_components=3)`
- `Z = pca.fit_transform(X_scaled)`

**Expected outcome:** `Z.shape == (995, 3)`. `pca.explained_variance_ratio_` contains three values telling you how much standardized variance is captured by each component.

**What this teaches:** Difference between the original high-dimensional features, PCA **scores**, and PCA **components**.

1. Standardize the feature vectors by removing the mean and scaling to unit variance.
2. Transform the data in $\mathbf{X}$ to a 3-dimensional subspace using [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html). Store the transformed data in a variable named `Z` representing the $995 \times 3$ transformed aberrations matrix $\mathbf{Z}$.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# TODO: Standardize X and store the result in X_scaled.
# TODO: Fit PCA with 3 components; keep the fitted model in pca.
# TODO: Store the projected samples in Z.

# --- Solution (uncomment to check) ---
# X_scaled = StandardScaler().fit_transform(X)
# pca = PCA(n_components=3)
# Z = pca.fit_transform(X_scaled)
# print("Projected shape:", Z.shape)
# print("Variance ratio:", pca.explained_variance_ratio_)
# assert Z.shape == (995, 3)

##### ❓ Population clustering

### ❓ Exercise 4 — Visualize populations with PC1 vs PC2

**Task:** Plot each population in its own color, using `Z[:, 0]` for PC1 and `Z[:, 1]` for PC2. Complete the plotting loop in the original code.

**Hints:**
- `populations == label` makes a Boolean mask for the current population.
- `Z[mask, 0]` and `Z[mask, 1]` select the appropriate coordinates.
- `plt.scatter(..., label=label, color=color, s=12, alpha=0.65)` makes a readable scatter plot.

**Expected outcome:** A 2D scatter plot with seven color-coded populations and a legend. Look for **overlap** as well as separation: no clean boundary is guaranteed.

**What this teaches:** Connecting unlabeled PCA coordinates to known metadata *after* PCA and interpreting clustering cautiously.

**Reflect:** Which populations overlap most? Which seem most separated? Does a scatter-plot cluster prove a discrete biological boundary? (**No**.)

We expect that different individuals within a population have similar values of their projections onto the principal components.

Complete the following code to produce a scatter plot of all feature vectors projected onto the first and second principal components (i.e., $\mathbf{Z}$).
Use the provided colors to visualize the different population clusters.

> Notice how individuals in a particular population have similar aberrations such that their projected feature vectors start to look similar.
> Also, note that individuals in different populations have aberrations at different loci.
>
> Furthermore, note how the clusters for the two Nigerian populations (ESN, YRI) largely overlap.
> The populations from Sierra Leone (MSL) and The Gambia (GWD) are separated from the Nigerian populations through the second principal component, whereas the first principal component distinguishes the Kenyan (LWK) population from the Nigerian populations.
>
> In summary, although the principal components do not entirely correlate with well-known geographical directions (North-South, East-West) as in the European example above, they clearly convey some kind of geographic information.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
for label, color in zip(
    np.unique(populations),
    (
        "tab:blue",
        "tab:orange",
        "tab:green",
        "tab:red",
        "tab:purple",
        "tab:brown",
        "tab:pink",
    ),
):
    # TODO: Select rows for this population and scatter PC1 against PC2.
    pass  # Remove this line when you uncomment the solution.
    # --- Solution (uncomment to check) ---
    # mask = populations == label
    # plt.scatter(Z[mask, 0], Z[mask, 1], label=label, color=color, s=12, alpha=0.65)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(loc="lower right")
plt.show()

##### ❓ Sex clustering

### ❓ Exercise 5 — Explore sex labels with PC1 vs PC3

**Task:** Repeat the plot, using **PC1** and **PC3**, colored by the two sex labels in the supplied metadata.

**Hints:** `Z[:, 2]` is PC3, because Python counts columns starting at zero. Filter with `sexes == label`.

**Expected outcome:** A scatter plot with two legend entries. In this supplied dataset, you may see separation along PC3 associated with the recorded sex labels.

**What this teaches:** The same PCA embedding can reveal variation related to a different annotation; PC indices are not biological meanings by themselves.

**Reflect:** Can we conclude from this plot alone that PC3 measures biological sex or that a particular chromosome caused it? (**No**; that requires information and checks beyond this visualization.)

As seen above, different individuals within a population have similar values of their projections onto the first two principal components.
Let us now inspect the third principal component.

Complete the following code to produce a scatter plot of all feature vectors projected onto the first and _third_ principal components.
Use the provided colors to visualize the different sexes.

> Interestingly enough, the third principal component seems to encode the sex information!

In [ ]:
plt.figure(figsize=(6, 4))
for label, color in zip(np.unique(sexes), ("tab:blue", "tab:orange")):
    # TODO: Scatter PC1 vs PC3 for this sex label.
    pass  # Remove this line when you uncomment the solution.
    # --- Solution (uncomment to check) ---
    # mask = sexes == label
    # plt.scatter(Z[mask, 0], Z[mask, 2], label=label, color=color, s=12, alpha=0.65)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 3")
plt.legend(loc="lower right")
plt.show()

##### ❓ A deeper look into the data

### ❓ Exercise 6 — Which SNP features drive PC3?

**Task:** Plot the 10,101 coefficients of `pca.components_[2]` (the **third** PCA component) against SNP feature index. Look for large **absolute** coefficients, because positive *and* negative values matter.

**Hints:** `np.arange(len(pca.components_[2]))` creates the x-axis; `plt.scatter(..., s=4)` is sufficient. `pca.components_[2]` has 10,101 coefficients, one per input SNP feature.

**Expected outcome:** A plot of PC3 component coefficients indexed from `0` to `10100`. The original notebook anticipates a concentration of larger coefficients near the end of the feature list.

**What this teaches:** PCA scores position individuals; a PCA **loading/component coefficient** shows how a feature contributes to a direction of variation.

**Interpret carefully:** The original exercise proposes that late-index SNPs might be sex-chromosome related. However, it also states that **genomic loci are unknown** in this dataset. We therefore cannot verify chromosome identity from this plot alone; a SNP-to-chromosome annotation would be needed.

We just remarked above that the third principal component is responsible for popping out the sex differences.
But can we say more? Can we say which aberrations are different between males and females?

To answer this, we can look at the principal axes (i.e., eigenvectors) in the feature space, representing the directions of maximum variance in the data.

Make a scatter plot of the 10,101 values of the third principal axe (accessible as `pca.components_[2]`) versus the indexes of the aberrations (i.e., a range from 0 to 10,100).

> We can observe that the SNPs at the highest approximately 500 indexes are responsible for encoding the sex differences.
> This aligns with the fact that in most bioinformatics applications the allosomes are listed as last chromosomes in a karyogram.

In [ ]:
# TODO: Save the PC3 component coefficients in pc3_loadings.
# --- Solution (uncomment to check) ---
# pc3_loadings = pca.components_[2]

plt.figure(figsize=(8, 4))
# TODO: Scatter SNP index against the corresponding PC3 coefficient.
# --- Solution (uncomment to check) ---
# plt.scatter(np.arange(len(pc3_loadings)), pc3_loadings, s=4)
plt.xlabel("SNP Index")
plt.ylabel("Principal Component 3")
plt.show()

## 4. Wrap-up: check your understanding

Answer these questions in your own words (no coding required):

1. **Why remove ID, sex, and population labels before PCA?** Because these are metadata, not the intended SNP features; including them would change the problem and potentially leak labels into the visualization.
2. **What does `X[i, j] = 1` mean?** Individual `i` has a base different from the dataset consensus at SNP locus `j`.
3. **What is the shape of `Z`, and what do its columns mean?** `(995, 3)`; the PC1–PC3 scores for the 995 individuals.
4. **Is PCA the same as a clustering algorithm?** No. PCA makes a low-dimensional representation; any clusters are patterns we inspect in that space.
5. **Do high PC3 coefficients identify chromosome names?** No. They identify influential *feature indices*. Chromosome annotation would be required.

### Optional self-check (after completing the exercises)
The following cell tests data structure and intermediate arrays; it does **not** score your biological interpretation.

In [ ]:
# Optional checks. Run only after completing Exercises 1–3.
assert consensus_bases.shape == (10101,), "Check Exercise 1"
assert X.shape == (995, 10101), "Check Exercise 2"
assert np.all((X == 0) | (X == 1)), "X must be binary"
assert Z.shape == (995, 3), "Check Exercise 3"
assert pca.components_.shape == (3, 10101), "PCA components must align with SNP features"
print("Structure checks passed. Now interpret the plots!")